# German argument-extraction head — `train_args2_de`

Trains on the **SALSA** German corpus with the **`deepset/gbert-large`** backbone.

### One-time setup (on your own machine)
The SALSA corpus is licence-restricted, so it can't live in a public repo — you upload it (plus the code) to *your* Drive once, as a single zip:

```bash
cd ~/Desktop/Academic_projects/Texture_Frames
zip -r texture_frames_colab.zip \
    encoder_parser \
    German_parser/*.py \
    German_parser/extracted/salsa_release.xml \
    German_parser/extracted/salsa_frames.xml
```
Then upload `texture_frames_colab.zip` to the **top level of your Google Drive** (`MyDrive/`).

### Runtime
`Runtime → Change runtime type → GPU`. **A100** fits `batch_size=16`; on **L4/T4** use `batch_size=8`.

### Crash safety
Checkpoints are written to **your Drive** every epoch. **If Colab disconnects, just re-run the Train cell** — it auto-resumes from the last epoch on Drive.

In [ ]:
!nvidia-smi

## 1. Mount Drive & unpack the project

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os
ZIP = "/content/drive/MyDrive/texture_frames_colab.zip"
assert os.path.exists(ZIP), f"Upload the project zip to {ZIP} first (see the intro cell)."
!rm -rf /content/Texture_Frames && mkdir -p /content/Texture_Frames
!unzip -q "$ZIP" -d /content/Texture_Frames

base = "/content/Texture_Frames"
need = ["German_parser/salsa_loader.py", "German_parser/salsa_lexicon.py",
        "encoder_parser/model_frame2.py", "encoder_parser/model_args2.py",
        "German_parser/extracted/salsa_release.xml",
        "German_parser/extracted/salsa_frames.xml"]
missing = [p for p in need if not os.path.exists(os.path.join(base, p))]
print("MISSING:" , missing or "none — all code + data present")
assert not missing, "Re-make the zip; some files are missing."

In [ ]:
!pip install -q "transformers==4.57.6" "accelerate>=0.30" sentencepiece simplemma "numpy>=2"

In [ ]:
import os, sys
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"
sys.path.insert(0, "/content/Texture_Frames/encoder_parser")
sys.path.insert(0, "/content/Texture_Frames/German_parser")
os.chdir("/content/Texture_Frames/German_parser")

import numpy, torch, transformers
print("numpy", numpy.__version__, "| torch", torch.__version__,
      "| transformers", transformers.__version__, "| cuda", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU — set Runtime → GPU."

## 2. Sanity check — a gold role span round-trips through the tokenizer

In [ ]:
from transformers import AutoTokenizer
from salsa_lexicon import SalsaLexicon
from salsa_loader import load_args_examples
from salsa_args_data import build_args_input, frame_fe_hint, remap_fe_span
from args2_data import gold_span_token_indices
from args_data import _clean_span_text

tok = AutoTokenizer.from_pretrained("deepset/gbert-large")
tok.add_special_tokens({"additional_special_tokens": ["<t>", "</t>"]})
lex = SalsaLexicon()

text, loc, frame, fes = [e for e in load_args_examples("dev", drop_unannotated=True) if e[3]][0]
hint = frame_fe_hint(lex, frame)
combined, prefix_len, ts, te = build_args_input(text, frame, loc, hint)
remapped = [(*remap_fe_span(s, e, ts, te, prefix_len), n) for n, s, e in fes]
enc = tok(combined, truncation=True, max_length=320, return_offsets_mapping=True)
om = [tuple(x) for x in enc["offset_mapping"]]
print("frame:", frame)
for a, b, name in gold_span_token_indices(om, remapped, prefix_len, combined):
    print(f"  {name:16} -> {_clean_span_text(combined[om[a][0]:om[b][1]])!r}")

## 3. Train  →  checkpoints to Drive (resumable)

~30–60 min on A100 (this head is heavier). Re-run after any disconnect to resume.

Pass `keep_discontinuous=False` to train only on contiguous role spans (the 13.8%-discontinuous-span experiment).

In [ ]:
import train_args2_de

CKPT_DIR = "/content/drive/MyDrive/Texture_Frames/checkpoints/args2_de"
print("checkpointing to Drive:", CKPT_DIR)

model, tokenizer, lexicon, role2id, id2role = train_args2_de.train(
    base_model="deepset/gbert-large",
    output_dir=CKPT_DIR,       # on Drive → survives disconnects
    epochs=5,
    batch_size=16,             # L4/T4: set to 8
    lr=1e-5,
    max_length=320,
    role_lambda=1.0,
    keep_discontinuous=True,   # False = contiguous-span-only experiment
    resume=True,
)

## 4. Evaluate — NULL-bias sweep, weighted F1 (non-core = 0.5)

In [ ]:
from train_args2_de import evaluate_args2_de, print_report

print("### DEV — pick the winning NULL-bias ###")
print_report(evaluate_args2_de(model, tokenizer, lexicon, role2id, id2role, split="dev"))

print("\n### TEST — report the dev-chosen NULL-bias ###")
print_report(evaluate_args2_de(model, tokenizer, lexicon, role2id, id2role, split="test"))

## 5. Save the final model to Drive

In [ ]:
import os, shutil
MODEL_DIR = "/content/drive/MyDrive/Texture_Frames/models/args2_de"
os.makedirs(MODEL_DIR, exist_ok=True)

for f in ["args2_model.pt", "role2id.json"]:
    shutil.copy(os.path.join(CKPT_DIR, f), MODEL_DIR)
tokenizer.save_pretrained(MODEL_DIR)

print("saved to:", MODEL_DIR)
for f in sorted(os.listdir(MODEL_DIR)):
    print(f"  {f:28} {os.path.getsize(os.path.join(MODEL_DIR, f))/1e6:8.1f} MB")

## Notes

- **Disconnected?** Re-run cells 1 → 3. The Train cell resumes from the last checkpoint on Drive.
- **Reclaim space:** after cell 5, you may delete `checkpoints/args2_de/` — keep `models/args2_de/`.
- Best NULL-bias should be **picked on dev, reported on test** (the sweep prints both).